# 10 — Advanced Physiology Features

Three literature-based additions:

**Option A:** Wang-Engel (1998) forward simulation → single feature `WE_predicted_flag_doy`
- Full physics: f(T) × f(V) × f(P) accumulated from sowing
- Flag leaf ≈ Sdev 0.65 (Zadoks GS 37-39 equivalent)

**Option B:** Emergence-based vernalization (correct starting point)
- Use Field emergence DOY when available (fallback Sep 1 previous year)
- Features: `emergence_doy`, `VD_from_emergence`, `fV_from_emergence`

**Option E:** Climate extreme events
- `heat_days_pre_SOS` (Tmax > 30°C)
- `cold_days_pre_SOS` (Tmin < -10°C)
- `temp_range_greenup` (Tmax_max − Tmin_min during green-up)

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

FEAT_PATH = 'data/processed/features/features_phenometrics.parquet'
PHENO_PATH = 'data/processed/buffer_300m/wheat_hrw_phenology_buffer_matched.parquet'
WEATHER_PATH = 'data/raw/satellite/daymet_daily_weather.csv'
MAP_PATH = 'data/processed/csb_to_buf_mapping.csv'
OUT_PATH = 'data/processed/features/features_with_aptt.parquet'

feat = pd.read_parquet(FEAT_PATH)
print(f'Features: {feat.shape}')
print(f'Columns: {feat.columns.tolist()[-10:]}')

## 1. Load & prepare weather + phenology

In [ ]:
# Weather
wx = pd.read_csv(WEATHER_PATH)
wx['date'] = pd.to_datetime(wx['date'], format='mixed')
wx['year'] = wx['date'].dt.year
wx['doy'] = wx['date'].dt.dayofyear
if 'tmin' in wx.columns:
    wx = wx.rename(columns={'tmin':'Tmin','tmax':'Tmax'})
wx['FIELDID'] = wx['FIELDID'].astype(str)
wx['T_mean'] = (wx['Tmin'] + wx['Tmax']) / 2
print(f'Weather: {len(wx):,} rows, {wx["FIELDID"].nunique()} fields, {wx["year"].min()}–{wx["year"].max()}')

# CSB → BUF mapping
mapping = {}
if os.path.exists(MAP_PATH):
    m_df = pd.read_csv(MAP_PATH)
    mapping = dict(zip(m_df['field_id'].astype(str), m_df['nearest_BUF'].astype(str)))

# Phenology — extract emergence DOY per field-year
pheno = pd.read_parquet(PHENO_PATH)
EMERGENCE_STAGES = {'Seed', 'Seed Swell', 'Germinating', 'Preplant',
                    'Shoot - Emerging', 'Shoot', 'Emerging', 'Emerging - Seedling',
                    'Seedling', 'Seedling - 1 Leaf', '1 Leaf'}
# These represent sowing → first-leaf stages in field phenology dataset
em = pheno[pheno['growth_stage'].isin(EMERGENCE_STAGES)].copy()
# For winter wheat: emergence happens in fall (previous calendar year)
# The 'year' in the phenology dataset is the harvest year. Emergence should be DOY > 240 of year-1 OR DOY < 60 of year.
# We keep the earliest emergence observation per (field, year).
em_doy = em.groupby(['FIELDID','year'])['doy'].min().reset_index()
em_doy = em_doy.rename(columns={'doy':'emergence_doy'})
print(f'Emergence observations: {len(em_doy)} field-years')
print(f'Emergence DOY distribution: median={em_doy["emergence_doy"].median():.0f}, range {em_doy["emergence_doy"].min()}-{em_doy["emergence_doy"].max()}')

## 2. Option A: Wang-Engel forward simulation

For each field-year:
1. Start at sowing DOY (fallback: day 274 = Oct 1 of previous year)
2. Daily: compute f_T × f_V × f_P and accumulate into S_dev
3. Report DOY when S_dev crosses 0.65 (flag leaf threshold)

In [ ]:
# Wang-Engel parameters (Table 1, p.10)
R_DEV_MAX = 0.035  # per-day max development rate (calibrated so S_dev~1 at flowering)
TMIN_V, TOPT_V, TMAX_V = 0, 24, 35
TMIN_VN, TOPT_VN, TMAX_VN = 1.3, 4.9, 15.7  # Porter & Gawith 1999
VND = 22.0
VNB = 4.4
HC = 7.0  # critical photoperiod (hours)
OMEGA = 0.28  # photoperiod sensitivity
FLAG_LEAF_THRESHOLD = 0.65  # Sdev value corresponding to GS 37-39

def beta_fT(T, tmin, topt, tmax):
    if T <= tmin or T >= tmax: return 0.0
    alpha = np.log(2) / np.log((tmax - tmin)/(topt - tmin))
    x = (T - tmin); xopt = (topt - tmin)
    return max(0, (2 * x**alpha * xopt**alpha - x**(2*alpha)) / xopt**(2*alpha))

def photoperiod_hours(lat, doy):
    decl = 23.45 * np.sin(np.radians(360/365 * (doy - 81)))
    cos_ha = -np.tan(np.radians(lat)) * np.tan(np.radians(decl))
    return 2 * np.degrees(np.arccos(np.clip(cos_ha, -1, 1))) / 15

def simulate_wang_engel(wx_subset, lat, sowing_doy, sowing_year):
    """Forward simulation. wx_subset: daily weather sorted by date. Returns DOY when S_dev >= 0.65."""
    if wx_subset is None or len(wx_subset) == 0: return np.nan
    wx_subset = wx_subset.sort_values('date').reset_index(drop=True)
    vd_cum, Sdev = 0.0, 0.0
    flag_doy = np.nan
    for _, r in wx_subset.iterrows():
        # Daily vernalization rate (β-function)
        fvn = beta_fT(r['T_mean'], TMIN_VN, TOPT_VN, TMAX_VN)
        vd_cum += fvn
        # Streck 2003 vernalization factor
        fV = vd_cum**5 / (VND**5 + vd_cum**5)
        # Temperature response (vegetative)
        fT = beta_fT(r['T_mean'], TMIN_V, TOPT_V, TMAX_V)
        # Photoperiod factor (Wang-Engel Eq. 10, long-day plants)
        hp = photoperiod_hours(lat, r['doy'])
        fP = max(0, 1 - np.exp(-OMEGA * (hp - HC))) if hp > HC else 0.0
        # Daily dev rate
        r_dev = R_DEV_MAX * fT * fV * fP
        Sdev += r_dev
        if np.isnan(flag_doy) and Sdev >= FLAG_LEAF_THRESHOLD:
            flag_doy = r['doy'] if r['date'].year == sowing_year + (0 if sowing_doy > 200 else 1) else r['doy']
            # Always return DOY of current-year calendar
            flag_doy = r['date'].timetuple().tm_yday
            return flag_doy
    return flag_doy

# Compute for each field-year
print('Running Wang-Engel simulation per field-year...')
results_WE = []
field_years = feat[['field_id','year','flag_true_doy']].drop_duplicates().values

for idx, (fid, yr, true_doy) in enumerate(field_years):
    if pd.isna(true_doy): continue
    # Get lat from features parquet
    lat_val = feat[(feat['field_id']==fid) & (feat['year']==yr)]['latitude'].values
    if len(lat_val) == 0 or np.isnan(lat_val[0]): continue
    lat = lat_val[0]
    # Get emergence DOY if available
    em_match = em_doy[(em_doy['FIELDID']==fid) & (em_doy['year']==yr)]
    if len(em_match) > 0:
        sow_doy = int(em_match['emergence_doy'].iloc[0])
    else:
        sow_doy = 274  # Oct 1 fallback
    # Get weather: from sowing (previous year if sow_doy >= 200) through harvest year
    wx_fid = mapping.get(fid, fid)
    if sow_doy >= 200:
        # Sowing previous fall
        wx_sub = wx[(wx['FIELDID']==wx_fid) & 
                    (((wx['year']==yr-1) & (wx['doy']>=sow_doy)) | (wx['year']==yr))]
    else:
        # Emergence already in harvest year
        wx_sub = wx[(wx['FIELDID']==wx_fid) & (wx['year']==yr) & (wx['doy']>=sow_doy)]
    if len(wx_sub) == 0: continue
    pred = simulate_wang_engel(wx_sub, lat, sow_doy, yr)
    results_WE.append({'field_id':fid, 'year':yr, 'WE_predicted_flag_doy': pred,
                       'emergence_doy': sow_doy,  # All observed sowing/emergence dates (fall or spring)
                       'sowing_doy_used': sow_doy})
    if (idx+1) % 500 == 0:
        print(f'  {idx+1}/{len(field_years)} processed')

we_df = pd.DataFrame(results_WE)
print(f'\nDone: {len(we_df)} field-years')
print(we_df[['WE_predicted_flag_doy','emergence_doy']].describe())
# Correlation with true flag DOY
merged = we_df.merge(feat[['field_id','year','flag_true_doy']], on=['field_id','year'])
merged = merged.dropna(subset=['WE_predicted_flag_doy','flag_true_doy'])
r = np.corrcoef(merged['WE_predicted_flag_doy'], merged['flag_true_doy'])[0,1]
rmse = np.sqrt(np.mean((merged['WE_predicted_flag_doy']-merged['flag_true_doy'])**2))
print(f'\nWE prediction vs observed flag leaf: r={r:+.3f}, RMSE={rmse:.1f}d')

## 3. Option B: Emergence-based vernalization features

Compute VD and fV from actual emergence (or fallback Sep 1) instead of Jan 1.

In [ ]:
def vd_from_emergence(wx_subset, emergence_doy, sowing_year):
    """Compute cumulative VD and fV at each day starting from emergence."""
    if wx_subset is None or len(wx_subset) == 0: return None
    wx_sub = wx_subset.sort_values('date').reset_index(drop=True)
    wx_sub = wx_sub.copy()
    wx_sub['vd_daily'] = wx_sub['T_mean'].apply(
        lambda T: beta_fT(T, TMIN_VN, TOPT_VN, TMAX_VN))
    wx_sub['vd_cum_from_em'] = wx_sub['vd_daily'].cumsum()
    wx_sub['fV_from_em'] = wx_sub['vd_cum_from_em']**5 / (VND**5 + wx_sub['vd_cum_from_em']**5)
    return wx_sub

results_B = []
for idx, (fid, yr, true_doy) in enumerate(field_years):
    if pd.isna(true_doy): continue
    em_match = em_doy[(em_doy['FIELDID']==fid) & (em_doy['year']==yr)]
    em_doy_val = int(em_match['emergence_doy'].iloc[0]) if len(em_match) > 0 else 274
    sos_val = feat[(feat['field_id']==fid) & (feat['year']==yr)]['NDVI_SOS'].values
    if len(sos_val)==0 or np.isnan(sos_val[0]): continue
    sos = int(sos_val[0])
    wx_fid = mapping.get(fid, fid)
    if em_doy_val >= 200:
        wx_sub = wx[(wx['FIELDID']==wx_fid) & 
                    (((wx['year']==yr-1) & (wx['doy']>=em_doy_val)) |
                     ((wx['year']==yr) & (wx['doy']<=sos)))]
    else:
        wx_sub = wx[(wx['FIELDID']==wx_fid) & (wx['year']==yr) & 
                    (wx['doy']>=em_doy_val) & (wx['doy']<=sos)]
    out = vd_from_emergence(wx_sub, em_doy_val, yr)
    if out is None or len(out)==0: continue
    results_B.append({
        'field_id': fid, 'year': yr,
        'VD_from_emergence_at_SOS': out['vd_cum_from_em'].iloc[-1],
        'fV_from_emergence_at_SOS': out['fV_from_em'].iloc[-1],
        'days_emergence_to_SOS': len(out)
    })
    if (idx+1) % 500 == 0:
        print(f'  {idx+1}/{len(field_years)} processed')

b_df = pd.DataFrame(results_B)
print(f'\nOption B done: {len(b_df)} field-years')
print(b_df.describe().round(2))

## 4. Option E: Climate extreme events

In [ ]:
HEAT_THRESHOLD = 30.0   # °C
COLD_THRESHOLD = -10.0  # °C

results_E = []
for idx, (fid, yr, true_doy) in enumerate(field_years):
    if pd.isna(true_doy): continue
    sos_val = feat[(feat['field_id']==fid) & (feat['year']==yr)]['NDVI_SOS'].values
    pos_val = feat[(feat['field_id']==fid) & (feat['year']==yr)]['NDVI_POS'].values
    if len(sos_val)==0 or np.isnan(sos_val[0]): continue
    sos = int(sos_val[0])
    pos = int(pos_val[0]) if len(pos_val)>0 and not np.isnan(pos_val[0]) else sos+60
    wx_fid = mapping.get(fid, fid)
    wx_yr = wx[(wx['FIELDID']==wx_fid) & (wx['year']==yr)]
    if len(wx_yr) == 0: continue
    pre_sos = wx_yr[wx_yr['doy'] <= sos]
    greenup = wx_yr[(wx_yr['doy'] > sos) & (wx_yr['doy'] <= pos)]
    results_E.append({
        'field_id': fid, 'year': yr,
        'heat_days_pre_SOS': int((pre_sos['Tmax'] > HEAT_THRESHOLD).sum()),
        'cold_days_pre_SOS': int((pre_sos['Tmin'] < COLD_THRESHOLD).sum()),
        'temp_range_greenup': float(greenup['Tmax'].max() - greenup['Tmin'].min()) if len(greenup)>0 else np.nan,
        'max_Tmax_pre_SOS': float(pre_sos['Tmax'].max()) if len(pre_sos)>0 else np.nan,
        'min_Tmin_pre_SOS': float(pre_sos['Tmin'].min()) if len(pre_sos)>0 else np.nan,
    })
    if (idx+1) % 500 == 0:
        print(f'  {idx+1}/{len(field_years)} processed')

e_df = pd.DataFrame(results_E)
print(f'\nOption E done: {len(e_df)} field-years')
print(e_df.describe().round(2))

## 5. Merge all new features + save

In [ ]:
# Start from existing features
merged = feat.copy()
merged['field_id'] = merged['field_id'].astype(str)
merged['year'] = merged['year'].astype(int)

for df_new, name in [(we_df, 'Option A (WE)'), (b_df, 'Option B (emergence)'), (e_df, 'Option E (extremes)')]:
    if len(df_new) == 0:
        print(f'  {name}: SKIPPED (empty)')
        continue
    df_new['field_id'] = df_new['field_id'].astype(str)
    df_new['year'] = df_new['year'].astype(int)
    before = merged.shape[1]
    merged = merged.merge(df_new, on=['field_id','year'], how='left')
    added = merged.shape[1] - before
    print(f'  {name}: +{added} features, {df_new.shape[0]} rows')

print(f'\nFinal merged shape: {merged.shape}')
new_cols = [c for c in merged.columns if c not in feat.columns]
print(f'New features: {new_cols}')

# Correlations with target
print('\nCorrelations with flag_true_doy:')
for c in new_cols:
    sub = merged[[c, 'flag_true_doy']].dropna()
    if len(sub) < 100: 
        print(f'  {c:32s}  n={len(sub)} SKIP')
        continue
    r = np.corrcoef(sub[c], sub['flag_true_doy'])[0,1]
    print(f'  {c:32s}  n={len(sub):4d}  r={r:+.3f}')

merged.to_parquet(OUT_PATH, index=False)
print(f'\nSaved: {OUT_PATH}')

## 6. Quick Lasso run on augmented features

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

meta_cols = ['field_id','year','flag_true_doy','n_obs']
ndre = [c for c in merged.columns if c.startswith('NDRE')]
redund = ['GDD_M2_at_SOS', 'VD_at_SOS']
feat_cols_new = [c for c in merged.columns if c not in meta_cols and c not in ndre and c not in redund]
df2 = merged.dropna(subset=['flag_true_doy']).copy()
print(f'Samples: {len(df2)}, features: {len(feat_cols_new)}')

# LOYO CV
all_pred, all_true = [], []
for yr in sorted(df2['year'].unique()):
    tr = df2[df2['year']!=yr]; te = df2[df2['year']==yr]
    pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LassoCV(cv=5, max_iter=10000))])
    pipe.fit(tr[feat_cols_new], tr['flag_true_doy'])
    p = pipe.predict(te[feat_cols_new])
    all_pred.extend(p); all_true.extend(te['flag_true_doy'].values)
all_pred, all_true = np.array(all_pred), np.array(all_true)
rmse = np.sqrt(np.mean((all_pred-all_true)**2))
r2 = 1 - np.sum((all_pred-all_true)**2)/np.sum((all_true-all_true.mean())**2)
w10 = np.mean(np.abs(all_pred-all_true)<=10)*100
w15 = np.mean(np.abs(all_pred-all_true)<=15)*100
print(f'\nAugmented model (42 -> {len(feat_cols_new)} features):')
print(f'  RMSE = {rmse:.3f} d')
print(f'  R²   = {r2:.3f}')
print(f'  ±10d = {w10:.1f}%')
print(f'  ±15d = {w15:.1f}%')

# Feature importance
full_pipe = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()),
                     ('m', LassoCV(cv=5, max_iter=10000))])
full_pipe.fit(df2[feat_cols_new], df2['flag_true_doy'])
coefs = pd.DataFrame({'feat': feat_cols_new, 'coef': full_pipe.named_steps['m'].coef_,
                      'abs': np.abs(full_pipe.named_steps['m'].coef_)}).sort_values('abs', ascending=False)
print('\nTop 20 Lasso coefs:')
print(coefs.head(20).to_string(index=False))
print('\nNew features specifically:')
print(coefs[coefs['feat'].isin(new_cols)].to_string(index=False))